In [1]:
!pip install transformers datasets seqeval evaluate accelerate -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.6 MB/s eta 0:00:00


In [3]:
import numpy as np
import os
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import evaluate
from datasets import Dataset

# ---FİLE NAMES ---
TRAIN_FILE = "train_corrected.txt"
VALID_FILE = "valid_corrected.txt"
TEST_FILE = "test_corrected.txt"

MODEL_NAME = "indolem/indobert-base-uncased" #similar model to pos

# 1. TXT file read for ner
def load_ner_data(file_path):
    print(f"Reading: {file_path}...")
    sentences = []
    labels = []

    current_sentence = []
    current_labels = []

    # COLLECT ALL THE LABEL
    unique_tags = set()

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line: # EMPTY LİNE END OF LİNE
                if current_sentence:
                    sentences.append(current_sentence)
                    labels.append(current_labels)
                    current_sentence = []
                    current_labels = []
            else:
                # SPLİT LİNE BY SPACE
                parts = line.split()
                if len(parts) >= 2:
                    word = parts[0]
                    tag = parts[-1] #

                    current_sentence.append(word)
                    current_labels.append(tag)
                    unique_tags.add(tag)

        # ADD THE LAST SENT
        if current_sentence:
            sentences.append(current_sentence)
            labels.append(current_labels)

    return sentences, labels, sorted(list(unique_tags))

#LOAD FİLE
print("Data is being processed..")
train_s, train_l, tags_train = load_ner_data(TRAIN_FILE)
valid_s, valid_l, tags_valid = load_ner_data(VALID_FILE)
test_s, test_l, tags_test = load_ner_data(TEST_FILE)

# MERGE TAG LİST
label_list = sorted(list(set(tags_train) | set(tags_valid) | set(tags_test)))
print(f"Tags Found ({len(label_list)} piece): {label_list}")

# Convert to Dataset Format
def create_hf_dataset(sentences, labels):
    return Dataset.from_dict({"tokens": sentences, "ner_tags": labels})

train_dataset = create_hf_dataset(train_s, train_l)
val_dataset = create_hf_dataset(valid_s, valid_l)
test_dataset = create_hf_dataset(test_s, test_l)

# 2. Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

def align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

print("Tokenization is underway...")
tokenized_train = train_dataset.map(align_labels, batched=True)
tokenized_val = val_dataset.map(align_labels, batched=True)
tokenized_test = test_dataset.map(align_labels, batched=True)

# 3. Model Setup
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label_list), id2label=id2label, label2id=label2id
)

# 4. Metrics (Seqeval )
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# 5. Train settings
training_args = TrainingArguments(
    output_dir="indobert_ner_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Başlat
print("NER Training Begins")
trainer.train()

# Test Sonuçları
print("📊 Test Set Results")
results = trainer.evaluate(tokenized_test)
print(results)

Data is being processed..
Reading: train_corrected.txt...
Reading: valid_corrected.txt...
Reading: test_corrected.txt...
Tags Found (39 piece): ['B-CRD', 'B-DAT', 'B-EVT', 'B-FAC', 'B-GPE', 'B-LAN', 'B-LAW', 'B-LOC', 'B-MON', 'B-NOR', 'B-ORD', 'B-ORG', 'B-PER', 'B-PRC', 'B-PRD', 'B-QTY', 'B-REG', 'B-TIM', 'B-WOA', 'I-CRD', 'I-DAT', 'I-EVT', 'I-FAC', 'I-GPE', 'I-LAN', 'I-LAW', 'I-LOC', 'I-MON', 'I-NOR', 'I-ORD', 'I-ORG', 'I-PER', 'I-PRC', 'I-PRD', 'I-QTY', 'I-REG', 'I-TIM', 'I-WOA', 'O']
Tokenization is underway...


Map:   0%|          | 0/12514 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Map:   0%|          | 0/2520 [00:00<?, ? examples/s]

Map:   0%|          | 0/2397 [00:00<?, ? examples/s]

...

NER Training Begins


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.495900,0.191565,0.787044,0.837069,0.811286,0.944068
2,0.157600,0.178977,0.814219,0.846081,0.829844,0.948834
3,0.123500,0.183149,0.819689,0.850113,0.834624,0.949629


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


📊 Test Set Results


{'eval_loss': 0.17086099088191986, 'eval_precision': 0.8117891513560805, 'eval_recall': 0.8349831271091114, 'eval_f1': 0.8232228013751803, 'eval_accuracy': 0.9508277966206694, 'eval_runtime': 9.0366, 'eval_samples_per_second': 265.255, 'eval_steps_per_second': 16.599, 'epoch': 3.0}
